## B3a - Machine Learning Activity 1 - Training Your Classifier
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 20, 2026

### About: In this notebook you'll train a neural network to tell the difference between the red and blue images you collected in B2!
### Overall: We'll walk through the whole process: looking at your data, building a model, choosing how it learns, and watching it train.

Machine learning is a complicated task, fortunately there are several frameworks or software packages designed to make machine learning easier for students and professionals.
We're going to use a machine learning framework called PyTorch. PyTorch is an open-source machine learning library used for various tasks such as natural language processing, computer vision, and deep learning.

<img src="Graphics/Neural_net_classification.png">

### The diagram above shows the neural network created during training for a classification task. In the example, the model's task is to classify images as either a cat or a dog.
By looking at many labeled images, the model learns the small and large differences between the classes. This learning process requires lots of labeled images -- which is exactly what you collected in B2!

## In this notebook we'll train something similar: our classifier will learn to tell "**red**" and "**blue**" images apart.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

# Importing our training utilities -- these handle the data loading, model
# building, and training loop so we can focus on understanding what's
# happening at each step, rather than getting lost in the code that makes
# it happen.
from train_utils import (
    preview_dataset,
    prepare_dataloaders,
    build_model,
    train_model,
)

### Before we train anything, let's look at the data we're about to use to teach our AI.
### This is always worth doing first, it's much faster to notice a problem (a missing class, a mislabeled folder, too few images) now than after waiting through a full training run.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

data_dir = "/home/explorer/Datasets/red_blue_dataset"

preview_dataset(data_dir)

### Take a look at the images and counts above.
Do the sample images actually match their class labels? Are the red/blue image counts roughly balanced, or is one class much bigger than the other?
**Real Data Science**: A model trained on an imbalanced dataset (say, 100 red images but only 10 blue) tends to get lazy, it can get a deceptively high accuracy just by guessing the bigger class most of the time.  
If your counts are very different, it's worth collecting a few more images of the smaller class before training.

## Choosing your hyperparameters
### A **hyperparameter** is a setting you choose *before* training starts, they have a real effect on the time it takes to train the model and the resulting accuracy of the model.
### This is your chance to adjust the outcome!

### Change these if you wnat, or just use the default. Later, you can re-run training with *different hyperparameters* and a new `model_name` to see what happens!

In [ ]:
##### ----- FEEL FREE TO CHANGE THESE VALUES ----- #####

# model_name: a unique name for this training run. Used to name your saved
# model and its training plot -- pick something you'll recognize later!
model_name = "red_blue_classifier_v1"

# epochs: how many times the model sees your *entire* training set.
# Too few epochs and the model won't have learned much yet (underfitting).
# Too many epochs and the model can start memorizing your training images
# instead of learning general red-vs-blue differences (overfitting) --
# watch the training plot for test accuracy flattening out or dropping.
epochs = 15

# learning_rate: how big a step the model takes to correct its mistakes
# after each batch of images. Too high and training can bounce around
# without ever settling down. Too low and training crawls, needing far
# more epochs to get anywhere.
learning_rate = 0.001

# momentum: how much of the previous step's direction carries into the
# next step. Higher momentum smooths out noisy, back-and-forth updates,
# but too much can cause training to overshoot a good solution.
momentum = 0.9

### ⚙️ Advanced, optional setting: `batch_size`
### `batch_size` controls how many images the model looks at together before updating itself. Larger batch sizes usually train faster and more smoothly -- but our robots' Jetson Orin Nano only has 4 GB of memory to work with. If `batch_size` is too large (32, 64, 128...), especially with other notebooks or programs open, training can crash with a memory error.
### We're keeping `batch_size` separate from the hyperparameters above and defaulting it to a safe value. If you want to experiment, try 8 or 16 -- but if you hit a memory error, lower it back down.

In [ ]:
##### ----- YOU CAN CHANGE THIS, BUT SEE THE NOTE ABOVE FIRST ----- #####
batch_size = 8

## Building your dataset
### Now that we've chosen our hyperparameters, let's actually load the images into PyTorch and split them into a **training set** (used to teach the model) and a **test set** (used to check what it's learned, using images it never trained on).

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

train_loader, test_loader, class_names = prepare_dataloaders(
    data_dir,
    batch_size=batch_size,
    test_fraction=0.2
)

## Building your model
### We're using a model called `resnet18`, which has already been trained on millions of images and learned to recognize many general shapes, colors, and textures.  
### Using a pretrained model as the basis for our new model is called **transfer learning,** instead of starting from scratch, our model starts from everything it already knows and only has to learn what makes *your* classes different.
### `resnet18` was originally trained to sort images into 1000 categories. We'll swap out its final layer for a new one sized to however many classes you collected -- in this case, 2 (red and blue).

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

model, device = build_model(num_classes=len(class_names))

## Training your model
### Now for the main event! During training, the model will repeatedly look at your training images, guess a label, check how wrong it was, and adjust itself slightly to do better next time. After each full pass (epoch), we'll check its accuracy on the test images it hasn't seen.
### Watch the printed accuracy as it trains -- and once it's done, you'll see a plot summarizing the whole run.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

history, training_record = train_model(
    model,
    train_loader,
    test_loader,
    device,
    class_names,
    data_dir,
    model_name,
    epochs,
    learning_rate,
    momentum,
)

## Reading your results
### Look at the plot above.
### Does test accuracy keep climbing as training goes on, or does it flatten out (or even start dropping) partway through? If training loss keeps falling while test accuracy stalls, that's a sign of **overfitting** -- your model is starting to memorize the training images rather than learning general red-vs-blue differences.
### 🤖 Real Robotics Engineering: engineers rarely get a great model on the first try. Comparing runs -- more epochs, a different learning rate, more collected data -- is a normal, expected part of the process, not a sign something went wrong.

## Optional but valuable: train a second model
### **This step is optional if we're short on time today** -- but if you have a few extra minutes, it's one of the best ways to build real intuition about hyperparameters.
### Scroll back up to the hyperparameters cell, change `model_name` to something new (like `"red_blue_classifier_v2"`) along with one other value -- maybe `epochs` or `learning_rate` -- then re-run the cells below it. Compare the new plot to your first one. Which run did better, and why do you think that is?

## You should now see a `best_model_<your_model_name>.pth` file and a matching `.png` plot in your Models folder, along with an entry in `training_log.txt`.

## **NEXT**: Continue to B3b to prepare and use your trained model!